In [1]:
import importlib
from snowflake.snowpark.functions import col,trim,split,lit,split_part
from snowflake.snowpark.functions import col, sum as _sum, when, is_null
from snowflake.snowpark import functions as F
from snowflake.snowpark import Window
import sys 
sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils
import Snowflake_configuration
snowflake_conn_prop = Snowflake_configuration.ds1_role_json
from snowflake.snowpark.session import Session
import pandas as pd
import numpy as np
import datetime
from dateutil.relativedelta import relativedelta

In [2]:
session = Session.builder.configs(snowflake_conn_prop).create()
session.use_database('MOP_DATABASE')
session.use_schema('SOQ')

In [3]:
# --- CONFIGURATION ---
TRANSIT_TABLE_NAME = "SAP_DATA_DATABASE.SAP_DATA.ZSDTTPTACTFRT"
VSTEL_MAPPING = {"HHDS": "HHHD", "HHGS": "HHHG", "HHUS": "HHHU", "HS4N": "HM4N", "HS5V": "HM5V", "HS6C": "HM6C"}
CUSTOMER_TYPE_TO_CONSIDER = ['Individual'] 
SKU_SUPERCEDENCE_MODEL_FAMILY = 'MOP_DATABASE.SOQ.SKU_SUPERCEDENCE_MODEL_FAMILY_MAR_2026_UPDATED_V18' 
FESTIVE_TABLES = 'MOP_DATABASE.SOQ.FESTIVE_DAYS_SOQ' 
FESTIVE_PROPORTION_TABLE = 'WORK_DATABASE.MOP.FESTIVE_INDIAN_SEASON_AGG_MONTH' 
FINAL_TABLE = "MOP_DATABASE.SOQ.TRAIN_AND_TEST_DATA_FOR_TFT" 

### Step 1 

In [4]:
#Returns a list of model names or None based on user filter parameter settings.
#This helps us to filter out the models for which forecasting is not required
def return_models_for_forecasting(session, use_selected_models, logger):
    """
    Returns a list of model names or None based on user filter parameter settings.
    """
    if not use_selected_models:
        # logger.info("Model filter disabled — considering ALL models in ECR sales.")
        return None

    models_for_forecasting = session.table('MOP_DATABASE.SOQ.MODELS_FOR_FORECASTING').to_pandas()
    name_of_models = models_for_forecasting["MODEL_NAME"].tolist()
    # logger.info("Model filter enabled — %s models selected for forecasting.", len(name_of_models))
    return name_of_models

In [5]:
name_of_models = return_models_for_forecasting(session,True,None)

In [6]:
def fetchSKUSupercedence_snowpark(session, SKU_SUPERCEDENCE_MODEL_FAMILY):
    data = session.table("MOP_DATABASE.SOQ.SKU_SUPERCEDENCE") 
    data_1 = session.table("MOP_DATABASE.SOQ.MODEL_FAMILY_MAPPING") 
    result = data.join(data_1, on="MODEL", how="left") 
    result = result.with_column("SKU_UNIQUE_FAMILY_CODE", col("UNIQUEFAMILYCODE")) 

    for old_col in result.columns:
        new_col = old_col.replace('"','')
        result = result.rename(old_col, new_col) 
    
    result = result.with_column("MODEL_FAMILY_CODE",
                            F.concat(F.col("MODEL_FAMILY"), F.lit('<>'),
                            F.substring(F.col("UNIQUEFAMILYCODE"),
                                F.charindex(F.lit('<>'), F.col("UNIQUEFAMILYCODE")) + F.lit(2))
                                )) 
    
    result = result.rename("UNIQUEFAMILYCODE", "UNIQUE FAMILY CODE") 
    result.write.mode("overwrite").save_as_table(SKU_SUPERCEDENCE_MODEL_FAMILY) 
    return result


In [7]:
result = fetchSKUSupercedence_snowpark(session,SKU_SUPERCEDENCE_MODEL_FAMILY)

In [8]:
def get_ecr_sales_snowpark(session, customer_types, start_date, name_of_models,end_date):
    ecr_sales = session.table("ANALYTICS_DATABASE.ANALYTICS_SALES.CUSTOMER_RETAILS") \
        .filter(col("X_CUSTOMER_TYPE").in_(customer_types)) \
        .filter((col("CAL_DATE") >= F.lit(start_date)) & (col("CAL_DATE") <= F.lit(end_date)))
        
    if name_of_models is not None:
        ecr_sales = ecr_sales.filter(col("MODEL").isin(name_of_models))
        
    ecr_sales = ecr_sales.with_column("NET_SALES", 
    F.when(
        (col("INVOICED_SALES") + col("CANCELLED_SALES") + col("RETURNED_SALES")) < 0, 
        F.lit(0)
    ).otherwise(
        col("INVOICED_SALES") + col("CANCELLED_SALES") + col("RETURNED_SALES")
    )
)


    return ecr_sales

In [9]:
# ecr_sales = get_ecr_sales_snowpark(session,CUSTOMER_TYPE_TO_CONSIDER,'2023-04-01',name_of_models)

In [10]:
run_date = datetime.datetime.today()


In [11]:
def process_ecr_aggregation_snowpark(session, agg_type, customer_types,run_date, name_of_models, SKU_SUPERCEDENCE_MODEL_FAMILY):
    start_date = (run_date + relativedelta(day=1, months=-3)).date()

    end_date = start_date + relativedelta(months=3, days=-1)
    ecr_sales = get_ecr_sales_snowpark(session, customer_types, start_date, name_of_models,end_date)
    
    obd_data = session.table("MOP_DATABASE.SOQ.OBD2_MAPPING_VIEW") 
    sku_supercedence = session.table("MOP_DATABASE.SOQ.SKU_SUPERCEDENCE")
    
    obd_data_joined = obd_data.join(
        sku_supercedence.select("SKU", "SKUSTATUS"), 
        obd_data["CURRENT_OBD_SKU"] == sku_supercedence["SKU"], 
        how='left'
    )
    
    obd_data_active_skus = obd_data_joined.filter(F.lower(F.col("SKUSTATUS")) == 'active') \
                                          .select("CURRENT_OBD_SKU", "PREVIOUS_OBD_SKU")

    ecr_sales = ecr_sales.join(obd_data_active_skus, ecr_sales["SKU"] == obd_data_active_skus["PREVIOUS_OBD_SKU"], how="left")
    ecr_sales = ecr_sales.with_column("SKU", F.coalesce(col("CURRENT_OBD_SKU"), col("SKU"))) 
    
    sku_map = session.table(SKU_SUPERCEDENCE_MODEL_FAMILY).filter(F.lower(F.col("SKUSTATUS"))=='active')
    ecr_sales = ecr_sales.join(sku_map, ["MODEL", "SKU"], how="inner")
    
    parent_map = session.table("FIVETRAN_DATABASE.ORACLE_LDP_OLAP_SCHEMA.WC_INT_ORG_DH") \
        .select(col("X_DEALER_CODE_HIER").alias("DEALER_CODE"), col("PAR_ORG_NAME")).distinct() 

    ecr_sales = ecr_sales.join(parent_map, "DEALER_CODE", how="left") 
    ecr_sales = ecr_sales.with_column("PARENT_DEALER_CODE", split_part(col("PAR_ORG_NAME"), F.lit("-"), F.lit(1))) 

    ecr_sales = ecr_sales.distinct()
    
    if agg_type == "monthly":
        final_agg = ecr_sales.group_by(
            "PARENT_DEALER_CODE","MODEL_FAMILY_CODE","SKU",
            F.year("CAL_DATE").alias("CAL_YEAR"), 
            F.month("CAL_DATE").alias("CAL_MONTH")
        ).agg(F.sum("NET_SALES").alias("MONTHLY_DEALER_SKU_SALES")) 
        
        final_agg = final_agg.with_column("DATE", F.date_from_parts(col("CAL_YEAR"), col("CAL_MONTH"), F.lit(1))) 

    return final_agg.select("PARENT_DEALER_CODE","MODEL_FAMILY_CODE","SKU","DATE","MONTHLY_DEALER_SKU_SALES")


In [12]:
final_agg = process_ecr_aggregation_snowpark(session,"monthly",CUSTOMER_TYPE_TO_CONSIDER,run_date, name_of_models, SKU_SUPERCEDENCE_MODEL_FAMILY)

In [13]:
#final_agg is the Snowpark Dataframe - which has monthly sales at parent dealer code and SKU

In [14]:
final_agg.show()

-----------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"MODEL_FAMILY_CODE"                      |"SKU"           |"DATE"      |"MONTHLY_DEALER_SKU_SALES"  |
-----------------------------------------------------------------------------------------------------------------------------
|10221                 |XTREME 125<>DISC<>SELF<>CAST<>BLUE       |HXTRPISSCFICTB  |2026-05-01  |1.000000                    |
|12126                 |GLAMOUR<>DISC<>SELF<>CAST<>BLUE          |HGLMRDSSCFIMNB  |2026-06-01  |1.000000                    |
|10003                 |HF DELUXE<>DRUM<>SELF<>CAST<>BLUE        |HDLHADRSCFIBKB  |2026-06-01  |1.000000                    |
|10366                 |DESTINI<>DRUM<>SELF<>SHEET METAL<>WHITE  |HDESHDRLMFIPSW  |2026-04-01  |1.000000                    |
|11498                 |XTREME 125<>DRUM<>SELF<>CAST<>GREY       |HXTRSASSCFIBGY  |2026-06-01  |1.000000              

In [15]:
#Aggregation by dealer code and model family
agg1 = final_agg.group_by("PARENT_DEALER_CODE","MODEL_FAMILY_CODE","DATE").agg(F.sum(F.col("MONTHLY_DEALER_SKU_SALES")).alias("MONTHLY_DEALER_SKU_FAMILY_SALES"))
agg1.show()

------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"MODEL_FAMILY_CODE"                           |"DATE"      |"MONTHLY_DEALER_SKU_FAMILY_SALES"  |
------------------------------------------------------------------------------------------------------------------------
|11941                 |PASSION<>DRUM<>SELF<>CAST<>RED                |2026-05-01  |1.000000                           |
|12057                 |PLEASURE+<>DRUM<>SELF<>CAST<>BLUE             |2026-06-01  |1.000000                           |
|12280                 |HF DELUXE<>DRUM<>SELF<>CASTI<>BLACK           |2026-04-01  |1.000000                           |
|12095                 |SPLENDOR+<>DISC<>SELF<>CAST<>RED BLACK        |2026-06-01  |1.000000                           |
|11467                 |DESTINI<>DRUM<>SELF<>CAST<>RED                |2026-06-01  |1.000000                           |
|10356                 |PLEASURE

In [16]:
def remove_quotes_from_column_names(df):
    for old_col in df.columns:
        new_col = old_col.replace('"','')
        df = df.rename(old_col, new_col)
    return df

In [17]:
agg1 = remove_quotes_from_column_names(agg1)

In [18]:
agg1.filter(F.col("PARENT_DEALER_CODE")==12131).show()

-------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"MODEL_FAMILY_CODE"                            |"DATE"      |"MONTHLY_DEALER_SKU_FAMILY_SALES"  |
-------------------------------------------------------------------------------------------------------------------------
|12131                 |HF DELUXE<>DRUM<>SELF<>CAST<>BLACK             |2026-04-01  |22.000000                          |
|12131                 |PLEASURE+<>DRUM<>SELF<>CAST<>GREY              |2026-04-01  |1.000000                           |
|12131                 |SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK AND ACCENT  |2026-05-01  |80.000000                          |
|12131                 |PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY GREY   |2026-05-01  |6.000000                           |
|12131                 |SPLENDOR+<>DRUM<>SELF<>CAST<>RED BLACK         |2026-04-01  |2.000000                           |
|12131                 |

In [19]:
final_agg.filter(F.col("PARENT_DEALER_CODE")==12131).show()

-----------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"MODEL_FAMILY_CODE"                      |"SKU"           |"DATE"      |"MONTHLY_DEALER_SKU_SALES"  |
-----------------------------------------------------------------------------------------------------------------------------
|12131                 |PLEASURE+<>DRUM<>SELF<>CAST<>GREY        |HPLNCDRVCFIMVG  |2026-04-01  |1.000000                    |
|12131                 |DESTINI<>DRUM<>SELF<>CAST<>WHITE         |HDESYHSZCFIFWT  |2026-06-01  |1.000000                    |
|12131                 |HF DELUXE<>DRUM<>SELF<>CASTI<>BLUE       |HDLHAIRSCFIBKB  |2026-06-01  |0.000000                    |
|12131                 |SPLENDOR+<>DRUM<>SELF<>CAST<>GREY        |HSPUNHRSCFIIDG  |2026-05-01  |1.000000                    |
|12131                 |DESTINI<>DRUM<>SELF<>CAST<>WHITE         |HDSTMDRVCFIFWT  |2026-05-01  |1.000000              

In [20]:
#Final join for weight calculation
joined_df = final_agg.join(agg1,on=["PARENT_DEALER_CODE","MODEL_FAMILY_CODE","DATE"],how='inner')
joined_df.show()

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"MODEL_FAMILY_CODE"                            |"DATE"      |"SKU"           |"MONTHLY_DEALER_SKU_SALES"  |"MONTHLY_DEALER_SKU_FAMILY_SALES"  |
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------
|11036                 |XOOM<>DRUM<>SELF<>CAST<>RED                    |2026-05-01  |HXOMFDSSCFIBRD  |1.000000                    |1.000000                           |
|10719                 |SPLENDOR+<>DISC<>SELF<>CAST<>BLUE              |2026-04-01  |HSPLMTSSCFIBSB  |1.000000                    |1.000000                           |
|10307                 |DESTINI<>DRUM<>SELF<>SHEET METAL<>BLACK        |2026-05-01  |HDESHDRLMFIPBK  |1.000000                    |1.000000                     

In [21]:
# joined_df.filter(((F.col("PARENT_DEALER_CODE")==12145) & (F.col("MODEL_FAMILY_CODE")=='GLAMOUR<>DISC<>SELF<>CAST<>BLACK') & (F.col("DATE")=='2026-04-01'))).show()

In [22]:
#Check which MODEL_FAMILY_CODE has multiple SKUs
# count_of_skus_per_family = joined_df.group_by("MODEL_FAMILY_CODE").agg(F.count_distinct("SKU").alias("NUMBER_OF_SKUS_PER_FAMILY")).sort(F.col("NUMBER_OF_SKUS_PER_FAMILY").desc()).show()

In [23]:
sku_family_sales_3_months = joined_df.group_by("PARENT_DEALER_CODE","MODEL_FAMILY_CODE").agg(F.sum("MONTHLY_DEALER_SKU_FAMILY_SALES").alias("FAMILY_SALES_ACROSS_3_MONTHS"))
sku_family_sales_3_months.show()

-------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"MODEL_FAMILY_CODE"                                |"FAMILY_SALES_ACROSS_3_MONTHS"  |
-------------------------------------------------------------------------------------------------------------
|12205                 |HF DELUXE<>DRUM<>SELF<>CAST<>BLUE                  |1.000000                        |
|10267                 |DESTINI<>DRUM<>SELF<>SHEET METAL<>BLUE             |1.000000                        |
|11236                 |SUPER SPLENDOR<>DRUM<>SELF<>CAST<>BLUE             |2.000000                        |
|10957                 |XOOM<>DISC<>SELF<>CAST<>BLACK                      |0.000000                        |
|12201                 |SPLENDOR+<>DRUM<>SELF<>CASTI<>BLACK                |1.000000                        |
|11686                 |XTREME 125<>DISC<>SELF<>CAST<>BLACK                |1.000000                        |
|12118    

In [24]:
sku_sales_3_months = joined_df.group_by("PARENT_DEALER_CODE","MODEL_FAMILY_CODE","SKU").agg(F.sum("MONTHLY_DEALER_SKU_SALES").alias("SKU_SALES_ACROSS_3_MONTHS"))
sku_sales_3_months.show()

-----------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"MODEL_FAMILY_CODE"                      |"SKU"           |"SKU_SALES_ACROSS_3_MONTHS"  |
-----------------------------------------------------------------------------------------------------------------
|11316                 |SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK GREY  |HSPLMTRSCFITGB  |6.000000                     |
|10955                 |DESTINI<>DRUM<>SELF<>CAST<>RED           |HDESYDRVCFIBRD  |2.000000                     |
|10952                 |XTREME 160<>DRUM<>SELF<>CAST<>BLACK      |HXTRGPDSCFIMBS  |1.000000                     |
|10818                 |XTREME 160<>DRUM<>SELF<>CAST<>BLACK      |HXTRGPDSCFIMBS  |2.000000                     |
|12237                 |SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK GREY  |HSPLMTRSCFITGB  |4.000000                     |
|10214                 |DESTINI<>DISC<>SELF<>CAST<>RED           |HDSTMDSZCFIBRD  |1.000

In [25]:
snowflake_utils.shape_of_snowpark_df(sku_sales_3_months)

(61473, 4)

In [26]:
snowflake_utils.shape_of_snowpark_df(sku_family_sales_3_months)

(46317, 3)

In [27]:
_3_months_sales_df = sku_family_sales_3_months.join(sku_sales_3_months,on=["PARENT_DEALER_CODE","MODEL_FAMILY_CODE"],how='inner')
_3_months_sales_df.show()

--------------------------------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"MODEL_FAMILY_CODE"                      |"FAMILY_SALES_ACROSS_3_MONTHS"  |"SKU"           |"SKU_SALES_ACROSS_3_MONTHS"  |
--------------------------------------------------------------------------------------------------------------------------------------------------
|12279                 |SUPER SPLENDOR<>DISC<>SELF<>CAST<>BLACK  |1.000000                        |HSPSEDSSCFIGBL  |1.000000                     |
|17081                 |XPULSE<>DISC<>SELF<>SPOKE<>SILVER        |1.000000                        |HXPLBTDSSFIALS  |1.000000                     |
|11251                 |DESTINI<>DRUM<>SELF<>CAST<>WHITE         |2.000000                        |HDESYHSZCFIFWT  |1.000000                     |
|11711                 |SPLENDOR+<>DRUM<>SELF<>CAST<>BLUE        |1.000000                        |HSPLMTRSCFIBSB  |1.

In [35]:
_3_months_sales_df.filter(((F.col("MODEL_FAMILY_CODE")=='XTREME 125<>DISC<>SELF<>CAST<>BLACK') & (F.col("PARENT_DEALER_CODE")==11250))).select(F.sum("SKU_SALES_ACROSS_3_MONTHS")).show()

----------------------------------------
|"SUM(""SKU_SALES_ACROSS_3_MONTHS"")"  |
----------------------------------------
|147.000000                            |
----------------------------------------



In [31]:
_3_months_sales_df.group_by("PARENT_DEALER_CODE","MODEL_FAMILY_CODE").agg(F.count_distinct("SKU").alias("NUMBER_OF_DISTINCT_SKUS")).sort(F.col("NUMBER_OF_DISTINCT_SKUS").desc()).show()

------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"MODEL_FAMILY_CODE"                  |"NUMBER_OF_DISTINCT_SKUS"  |
------------------------------------------------------------------------------------------
|10571                 |XTREME 125<>DISC<>SELF<>CAST<>BLACK  |10                         |
|11030                 |XTREME 125<>DISC<>SELF<>CAST<>BLACK  |8                          |
|10082                 |XTREME 125<>DISC<>SELF<>CAST<>BLACK  |8                          |
|10375                 |XTREME 125<>DISC<>SELF<>CAST<>BLACK  |8                          |
|11706                 |XTREME 125<>DISC<>SELF<>CAST<>BLACK  |8                          |
|10977                 |XTREME 125<>DISC<>SELF<>CAST<>BLACK  |7                          |
|10035                 |XTREME 125<>DISC<>SELF<>CAST<>BLACK  |7                          |
|11913                 |XTREME 125<>DISC<>SELF<>CAST<>BLACK  |7                          |

In [36]:
from snowflake.snowpark.window import Window

In [37]:
_3_months_sales_df.show()

--------------------------------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"MODEL_FAMILY_CODE"                      |"FAMILY_SALES_ACROSS_3_MONTHS"  |"SKU"           |"SKU_SALES_ACROSS_3_MONTHS"  |
--------------------------------------------------------------------------------------------------------------------------------------------------
|12279                 |SUPER SPLENDOR<>DISC<>SELF<>CAST<>BLACK  |1.000000                        |HSPSEDSSCFIGBL  |1.000000                     |
|17081                 |XPULSE<>DISC<>SELF<>SPOKE<>SILVER        |1.000000                        |HXPLBTDSSFIALS  |1.000000                     |
|11251                 |DESTINI<>DRUM<>SELF<>CAST<>WHITE         |2.000000                        |HDESYHSZCFIFWT  |1.000000                     |
|11711                 |SPLENDOR+<>DRUM<>SELF<>CAST<>BLUE        |1.000000                        |HSPLMTRSCFIBSB  |1.

In [38]:
partition_window = Window.partition_by("PARENT_DEALER_CODE","MODEL_FAMILY_CODE")

# _3_months_sales_df.with_column("FAMILY_SALES_CUMULATIVE_3_MONTHS",F.sum(""))
